# Lab 2 — Prompt Agent

A **prompt agent** is a declarative agent (model + instructions) that Foundry hosts for you — no container, no Docker. In this lab you will:

1. Create a prompt agent with the `azure-ai-projects` SDK.
2. Invoke it through the OpenAI-compatible Responses API.
3. Add a new **version** with updated instructions.

> Make sure you completed **Lab 1** first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

load_dotenv()

project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1")

project = AIProjectClient(endpoint=project_endpoint, credential=DefaultAzureCredential())
AGENT_NAME = "labs-prompt-agent"

## 1. Create the prompt agent

`create_version` registers the agent definition (model + instructions). The first call creates version `1`.

In [ ]:
agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a concise, helpful assistant. "
            "Answer in no more than three sentences."
        ),
    ),
)
print(f"Created agent '{agent.name}' version {agent.version} (model: {model})")

## 2. Invoke the agent

We invoke the agent by reference through the OpenAI-compatible Responses API.

In [ ]:
openai_client = project.get_openai_client()

response = openai_client.responses.create(
    extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="In one sentence, what is Azure AI Foundry?",
)
print(response.output_text)

## 3. Add a new version

Re-running `create_version` with different instructions produces a new version. Foundry serves the latest active version automatically.

In [ ]:
agent_v2 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a pirate-themed assistant. Answer briefly and in pirate speak."
        ),
    ),
)
print(f"New version: {agent_v2.version}")

response = openai_client.responses.create(
    extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="Say hello.",
)
print(response.output_text)

Prompt agents are perfect when your logic is just *instructions + model*. When you need **custom code, tools, or RAG in your own container**, use a hosted agent — continue to **Lab 3**.